In [1]:
import pandas
import yaml
from pathlib import Path

In [2]:
fixtures = Path("igvf_mice/fixtures")

if not fixtures.exists():
    fixtures.mkdir()

In [3]:
model = "igvf_mice.librarybarcode"

In [4]:
def normalize_udi_to_code(value):
    prefix = "UDI_Plate_WT_"
    assert value.startswith(prefix)
    barcode_number = int(value[len(prefix):])
    return "UDI{:02}".format(barcode_number)

assert normalize_udi_to_code("UDI_Plate_WT_1") == "UDI01"
assert normalize_udi_to_code("UDI_Plate_WT_45") == "UDI45"

In [5]:
#KIT_INT_DICT = {'custom_1': 1, 'WT': 48, 'WT_mini': 12, 'WT_mega': 96}
#kit_n = KIT_INT_DICT[kit]
#if kit_n == 12:
#    bc_round_set = [['bc1','n24_v4'], ['bc2','v1'], ['bc3','v1']]
#if kit_n == 96:
#    bc_round_set = [['bc1','n192_v4'], ['bc2','v1'], ['bc3','v1']]
#if kit_n == 48:
#    bc_round_set = [['bc1','v2'], ['bc2','v1'], ['bc3','v1']]
#
#if kit == 'WT' and chemistry == 'v2':
#    bc_round_set = [['bc1', 'n96_v4'], ['bc2', 'v1'], ['bc3', 'v1']]


def load_parse_round1_barcode(reagent, start_pk, barcode_csv):
    for i, row in barcode_csv.iterrows():
        yield {
            "model": model,
            "pk": start_pk + i,
            "fields": {
                "reagent": reagent,
                "name": row["uid"],
                "code": row["well"],
                "i7_sequence": row["sequence"],
                "barcode_type": row["type"], # TODO what do the codes mean again T,R?
                "well_position": row["well"],
            }
        }

def load_single_illumina_barcode(reagent, start_pk, barcode_list):
    if not isinstance(barcode_list, pandas.DataFrame):
        barcode_list = pandas.DataFrame(barcode_list, columns=["code", "i7_sequence"])

    for i, row in barcode_list.iterrows():
        yield {
            "model": model,
            "pk": start_pk + i,
            "fields": {
                "reagent": reagent,
                "name": row["code"],
                "code": row["code"],
                "i7_sequence": row["i7_sequence"],
            }
        }
        
def load_udi_barcodes(reagent, start_pk):
    #udi_wt_reagent = models.LibraryConstructionReagent.objects.get(name="wt-mega", version="v2")
    udi_wt_barcodes = pandas.read_excel("https://support.parsebiosciences.com/hc/en-us/article_attachments/12988277964436")
    for i, row in udi_wt_barcodes.iterrows():
        yield {
            "model": model,
            "pk": start_pk + i,
            "fields": {            
                "reagent": reagent,
                "name": row["Index name"],
                "code": normalize_udi_to_code(row["Index name"]),
                "i7_sequence": row["i7_index"],
                "i5_sequence": row["i5_index"],
                "well_position": row["well_position"],
            }
        }

def load_barcodes():
    single_illumina_mega_subpool = [
        ("1", "CAGATC"),
        ("2", "ACTTGA"),
        ("3", "GATCAG"),
        ("4", "TAGCTT"),
        ("5", "ATGTCA"),
        ("6", "CTTGTA"),
        ("7", "AGTCAA"),
        ("8", "AGTTCC"),
        ("9", "GAGTGG"),
        ("10", "CCGTCC"),
        ("11", "GTAGAG"),
        ("12", "GTCCGC"),
        ("13", "GTGAAA"),
        ("14", "GTGGCC"),
        ("15", "GTTTCG"),
        ("16", "CGTACG"),
    ]
    single_illumina_regular_subpool = [
        ("1", "CAGATC"),
        ("2", "ACTTGA"),
        ("3", "GATCAG"),
        ("4", "TAGCTT"),
        ("5", "ATGTCA"),
        ("6", "CTTGTA"),
        ("7", "AGTCAA"),
        ("8", "AGTTCC"),
    ]

    fixture = []
    # The reagent field needs to match the primary key from the library_construction_reagent fixture
    mega_reagent = "wt-mega-v2"
    regular_reagent = "wt-v2"
    
    bc1_n192_v4 = pandas.read_csv("bc_data_n192_v4.csv")
    start_pk = 1
    fixture.extend(load_parse_round1_barcode(mega_reagent, start_pk, bc1_n192_v4))
    start_pk = 1 + len(fixture)
    fixture.extend(load_single_illumina_barcode(mega_reagent, start_pk, single_illumina_mega_subpool))
    start_pk = 1 + len(fixture)
    
    # Load WT regular v2 barcodes
    bc1_n96_v4 = pandas.read_csv("bc_data_n96_v4.csv")
    fixture.extend(load_parse_round1_barcode(regular_reagent, start_pk, bc1_n96_v4))
    start_pk = 1 + len(fixture)
    fixture.extend(load_single_illumina_barcode(regular_reagent, start_pk, single_illumina_regular_subpool))
    start_pk = 1 + len(fixture)
    
    fixture.extend(load_udi_barcodes(mega_reagent, start_pk))
    start_pk = 1 + len(fixture)

    fixture.extend(load_udi_barcodes(regular_reagent, start_pk))
    start_pk = 1 + len(fixture)    
    return fixture


barcodes = load_barcodes()
barcodes

[{'model': 'igvf_mice.librarybarcode',
  'pk': 1,
  'fields': {'reagent': 'wt-mega-v2',
   'name': 'pbs_1239',
   'code': 'A1',
   'i7_sequence': 'CATTCCTA',
   'barcode_type': 'T',
   'well_position': 'A1'}},
 {'model': 'igvf_mice.librarybarcode',
  'pk': 2,
  'fields': {'reagent': 'wt-mega-v2',
   'name': 'pbs_1205',
   'code': 'A2',
   'i7_sequence': 'CTTCATCA',
   'barcode_type': 'T',
   'well_position': 'A2'}},
 {'model': 'igvf_mice.librarybarcode',
  'pk': 3,
  'fields': {'reagent': 'wt-mega-v2',
   'name': 'pbs_1247',
   'code': 'A3',
   'i7_sequence': 'CCTATATC',
   'barcode_type': 'T',
   'well_position': 'A3'}},
 {'model': 'igvf_mice.librarybarcode',
  'pk': 4,
  'fields': {'reagent': 'wt-mega-v2',
   'name': 'pbs_1211',
   'code': 'A4',
   'i7_sequence': 'ACATTTAC',
   'barcode_type': 'T',
   'well_position': 'A4'}},
 {'model': 'igvf_mice.librarybarcode',
  'pk': 5,
  'fields': {'reagent': 'wt-mega-v2',
   'name': 'pbs_1218',
   'code': 'A5',
   'i7_sequence': 'ACTTAGCT',
  

In [6]:
with open(fixtures / "librarybarcode.yaml", "wt") as outstream:
    outstream.write(yaml.dump(barcodes))